# 10 — Transfer learning: freeze, replace, and fine-tune

**Level:** Intermediate · **Study time:** 75–100 minutes (including experiments)

**Prerequisites:** Notebooks 08–09.

## What you will be able to do
- Pretrain an encoder and reuse its features on a related task.
- Verify frozen parameters stay unchanged.
- Fine-tune with separate learning rates.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 09](09_cnn_digits.ipynb) · [Next: 11](11_embeddings_and_text.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Define related source and target tasks

We first learn to recognize digits, then reuse the encoder to classify parity (even or odd) with fewer labels. All source pretraining uses the training partition: validation images are never seen in pretraining. This small local pretraining exercise demonstrates the mechanics; it is not an ImageNet-pretrained model or evidence of universal transfer gains.


In [ ]:
import copy
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32) / 16
y = torch.tensor(digits.target, dtype=torch.long)
tr, va = train_test_split(np.arange(len(y)), test_size=0.25, random_state=SEED, stratify=y.numpy())
source_loader = DataLoader(TensorDataset(X[tr], y[tr]), batch_size=64, shuffle=True)
encoder = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
source = nn.Sequential(encoder, nn.Linear(32, 10))
optimizer = torch.optim.Adam(source.parameters(), lr=0.003)
for epoch in range(12):
    for xb, yb in source_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(source(xb), yb)
        loss.backward(); optimizer.step()
source.eval()
with torch.no_grad():
    print("Source validation digit accuracy:", (source(X[va]).argmax(1) == y[va]).float().mean().item())


## Step 3 — Freeze an encoder and learn a new head

`requires_grad_(False)` prevents parameter gradients. It is distinct from `eval()`, which changes layers such as dropout and batch normalization. Our encoder has neither, but we explicitly set it to eval to illustrate feature extraction practice.


In [ ]:
transferred_encoder = copy.deepcopy(encoder)
transferred_encoder.requires_grad_(False)
target_model = nn.Sequential(transferred_encoder, nn.Linear(32, 2))
small_tr = tr[:160]
target_y = y % 2
before = {name: p.detach().clone() for name, p in transferred_encoder.named_parameters()}
head_optimizer = torch.optim.Adam(target_model[1].parameters(), lr=0.03)
for epoch in range(160):
    target_model.train(); transferred_encoder.eval()
    head_optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(target_model(X[small_tr]), target_y[small_tr])
    loss.backward(); head_optimizer.step()
assert all(torch.equal(before[name], p) for name, p in transferred_encoder.named_parameters())
def parity_accuracy(model):
    model.eval()
    with torch.no_grad():
        return (model(X[va]).argmax(1) == target_y[va]).float().mean().item()
frozen_accuracy = parity_accuracy(target_model)
print("Frozen encoder validation parity accuracy:", frozen_accuracy)


## Step 4 — Unfreeze with a smaller encoder learning rate

Fine-tuning allows the representation to adapt. A smaller encoder learning rate reduces abrupt changes to learned features. Build a new optimizer that includes the newly trainable parameters; simply unfreezing cannot add them to the old head-only optimizer.


In [ ]:
transferred_encoder.requires_grad_(True)
fine_optimizer = torch.optim.Adam([
    {"params": transferred_encoder.parameters(), "lr": 0.0003},
    {"params": target_model[1].parameters(), "lr": 0.003},
])
for epoch in range(70):
    target_model.train()
    fine_optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(target_model(X[small_tr]), target_y[small_tr])
    loss.backward(); fine_optimizer.step()
fine_accuracy = parity_accuracy(target_model)
print("Fine-tuned validation parity accuracy:", fine_accuracy)
assert any(not torch.equal(before[name], p) for name, p in transferred_encoder.named_parameters())


## Your turn — practice before opening the solutions

1. Train a parity classifier from scratch with the same 160 examples for 230 full-batch updates.
2. Compare its validation accuracy with the frozen and fine-tuned models; explain why one seed is weak evidence.
3. Count trainable parameters after freezing the transferred encoder again.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
torch.manual_seed(SEED)
scratch = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2))
scratch_opt = torch.optim.Adam(scratch.parameters(), lr=0.003)
for _ in range(230):
    scratch_opt.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(scratch(X[small_tr]), target_y[small_tr])
    loss.backward(); scratch_opt.step()
print({"scratch": parity_accuracy(scratch), "frozen": frozen_accuracy, "fine_tuned": fine_accuracy})
transferred_encoder.requires_grad_(False)
assert sum(p.numel() for p in target_model.parameters() if p.requires_grad) == 32 * 2 + 2


## Before moving on

To extend this to a large pretrained vision model, use the official torchvision weight transforms, replace its head, and keep the target test data separate from every development step. Pretrained weights require an explicit download and a compatible torchvision installation.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Transfer learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)


[Course guide](../README.md) · [Previous: 09](09_cnn_digits.ipynb) · [Next: 11](11_embeddings_and_text.ipynb)
